### import module

In [1]:
import os, sys, django
from pathlib import Path

BASE_DIR = Path.cwd()  # 현재 노트북 실행 위치
# recommend_model.ipynb가 backend 폴더 안에 있다면 아래 줄은 없어도 되지만, 안전하게 넣음
sys.path.append(str(BASE_DIR))

os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "moathon.settings")
django.setup()

In [22]:
import random
import numpy as np
import pandas as pd
from django.conf import settings
from accounts.models import User, Moathon
from datetime import date
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    top_k_accuracy_score,
    log_loss,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)
import joblib

### 재현성 고정

In [23]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

### DB에서 데이터 로딩 

In [4]:
qs = (
    Moathon.objects
    .select_related("user", "product_option", "product_option__product")
    .values(
        "id", "user_id",
        # y
        "product_option__product_id",

        # Moathon 입력(=추론 시 폼으로 받을 값)
        "target_amount", "start_amount", "term_months", "purpose",

        # User 입력(=추론 시 DB에서 자동 반영할 값)
        "user__gender", "user__credit_score", "user__assets", "user__salary",
        "user__average_monthly_spend", "user__tender", "user__birth",
    )
)
df = pd.DataFrame.from_records(qs)

df.rename(columns={"product_option__product_id": "y_product_id"}, inplace=True)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 14 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   id                           40000 non-null  int64 
 1   user_id                      40000 non-null  int64 
 2   y_product_id                 40000 non-null  int64 
 3   target_amount                40000 non-null  int64 
 4   start_amount                 40000 non-null  int64 
 5   term_months                  40000 non-null  int64 
 6   purpose                      40000 non-null  object
 7   user__gender                 40000 non-null  object
 8   user__credit_score           40000 non-null  int64 
 9   user__assets                 40000 non-null  int64 
 10  user__salary                 40000 non-null  int64 
 11  user__average_monthly_spend  40000 non-null  int64 
 12  user__tender                 40000 non-null  object
 13  user__birth                  40

### 전처리 / feature engineering

In [6]:
def calc_age(birth):
    if pd.isna(birth):
        return np.nan
    today = date.today()
    return today.year - birth.year - ((today.month, today.day) < (birth.month, birth.day))
# age로 변환
df["age"] = df["user__birth"].apply(calc_age)
# 목표액까지 얼마남았는지
df["target_minus_start"] = df["target_amount"] - df["start_amount"]
df["target_minus_start"] = df["target_minus_start"].clip(lower=0)
# 목표액까지 비율
df["target_to_start_ratio"] = (df["target_amount"] + 1) / (df["start_amount"] + 1)
# 매달 저축해야하는 비용
df["required_monthly_saving"] = df["target_minus_start"] / df["term_months"].replace(0, np.nan)
# 월급 계산
df["monthly_income"] = df["user__salary"] / 12.0
# 남는 목돈
df["monthly_saving_capacity"] = df["monthly_income"] - df["user__average_monthly_spend"]


### Train/Valid split

In [14]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, valid_idx = next(gss.split(df, groups=df["user_id"].values))

train_df = df.iloc[train_idx].copy()
valid_df = df.iloc[valid_idx].copy()

### X,y 구성 / encoding

In [18]:
feature_cols_num = [
    "user__credit_score", "user__assets", "user__salary", "user__average_monthly_spend",
    "age", "target_amount", "start_amount", "term_months",
    "target_minus_start", "target_to_start_ratio",
    "required_monthly_saving", "monthly_income", "monthly_saving_capacity",
]

feature_cols_cat = ["user__gender", "user__tender", "purpose"]

train_labels = set(train_df["y_product_id"].unique())
valid_df_in = valid_df[valid_df["y_product_id"].isin(train_labels)].copy()

X_train = train_df[feature_cols_num + feature_cols_cat].copy()
X_valid = valid_df_in[feature_cols_num + feature_cols_cat].copy()

le = LabelEncoder()
y_train = le.fit_transform(train_df["y_product_id"].to_numpy())
y_valid = le.transform(valid_df_in["y_product_id"].to_numpy())

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]), feature_cols_num),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), feature_cols_cat),
    ]
)

### 평가지표 정의
- 확률의 절대값보다 랭킹 품질을 평가
- 상위 후보군 품질을 평가 

In [26]:
def hit_at_k(proba, y_true, k=3):
    # 각 샘플(행)에서 점수를 오름차순 정렬했을 때 상위 k개 반환
    topk = np.argsort(proba, axis=1)[:, -k:]
    # 정답 인덱스가 Top-k 안에 있으면 True 아니면 False 
    return np.mean([y_true[i] in topk[i] for i in range(len(y_true))])

def mrr_at_k(proba, y_true, k=10):
    # Mean Reciprocal Rank@k
    topk = np.argsort(proba, axis=1)[:, -k:][:, ::-1]
    rr = []
    for i, true in enumerate(y_true):
        hits = np.where(topk[i] == true)[0]
        rr.append(1.0 / (hits[0] + 1) if len(hits) else 0.0)
    return float(np.mean(rr))

def ndcg_at_k(proba, y_true, k=10):
    # binary relevance (정답 1개) 기준 NDCG@k
    topk = np.argsort(proba, axis=1)[:, -k:][:, ::-1]
    ndcg = []
    for i, true in enumerate(y_true):
        hits = np.where(topk[i] == true)[0]
        if len(hits) == 0:
            ndcg.append(0.0)
        else:
            rank = hits[0] + 1
            dcg = 1.0 / np.log2(rank + 1)
            idcg = 1.0  # 정답 1개면 이상적 DCG는 1
            ndcg.append(dcg / idcg)
    return float(np.mean(ndcg))

def coverage_at_k(proba, k=3):
    # 추천 다양성(커버리지): valid에서 Top-k로 추천된 클래스 종류 비율
    topk = np.argsort(proba, axis=1)[:, -k:]
    rec_items = np.unique(topk.reshape(-1))
    return len(rec_items)

In [19]:
def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 15.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "objective": "multi:softprob",
        "num_class": len(le.classes_),
        "tree_method": "hist",
        "random_state": SEED,
        "eval_metric": "mlogloss",
    }

    model = XGBClassifier(**params)

    pipe = Pipeline([
        ("prep", preprocess),
        ("xgb", model),
    ])

    pipe.fit(
        X_train, y_train,
        xgb__eval_set=[(preprocess.fit_transform(X_valid), y_valid)],
        xgb__verbose=False,
    )

    proba = pipe.predict_proba(X_valid)
    score = hit_at_k(proba, y_valid, k=3)
    return score  # Hit@3 최대화

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

best_params = study.best_trial.params
best_params

[I 2025-12-15 19:39:21,396] A new study created in memory with name: no-name-dc7321b7-2bdb-4b28-844b-c1d589e7d868
[I 2025-12-15 19:45:32,265] Trial 0 finished with value: 0.5067017412000501 and parameters: {'max_depth': 4, 'learning_rate': 0.10513422460379866, 'n_estimators': 624, 'subsample': 0.6295817042719001, 'colsample_bytree': 0.7793766007334468, 'min_child_weight': 1.8359929084190751, 'reg_lambda': 0.0027497375616894904, 'reg_alpha': 0.009417206757641771}. Best is trial 0 with value: 0.5067017412000501.
[I 2025-12-15 19:58:22,725] Trial 1 finished with value: 0.5668295127145183 and parameters: {'max_depth': 5, 'learning_rate': 0.020459347627252274, 'n_estimators': 1485, 'subsample': 0.6519706076153877, 'colsample_bytree': 0.6913824055312565, 'min_child_weight': 12.17844521488568, 'reg_lambda': 0.4855553486390839, 'reg_alpha': 3.382899851406174}. Best is trial 1 with value: 0.5668295127145183.
[I 2025-12-15 20:12:24,164] Trial 2 finished with value: 0.5400225479143179 and paramet

{'max_depth': 4,
 'learning_rate': 0.07423734763541161,
 'n_estimators': 419,
 'subsample': 0.807347217350573,
 'colsample_bytree': 0.7372178782429201,
 'min_child_weight': 4.846682493607758,
 'reg_lambda': 5.943465670569476,
 'reg_alpha': 1.193362670594393}

In [27]:
best_xgb = XGBClassifier(
    **best_params,
    objective="multi:softprob",
    num_class=len(le.classes_),
    tree_method="hist",
    random_state=SEED,
    eval_metric="mlogloss",
)

final_pipe = Pipeline([
    ("prep", preprocess),
    ("xgb", best_xgb),
])

final_pipe.fit(X_train, y_train)

valid_proba = final_pipe.predict_proba(X_valid)
y_pred = np.argmax(valid_proba, axis=1)

# labels 범위를 명시(= valid_proba의 열 순서와 일치)
labels_all = np.arange(valid_proba.shape[1])

# ===== 기본/랭킹 지표 =====
metrics = {
    "LogLoss": log_loss(y_valid, valid_proba, labels=labels_all),
    "Accuracy@1": accuracy_score(y_valid, y_pred),
    "TopKAcc@3": top_k_accuracy_score(y_valid, valid_proba, k=3, labels=labels_all),
    "TopKAcc@5": top_k_accuracy_score(y_valid, valid_proba, k=5, labels=labels_all),
    "Hit@3": hit_at_k(valid_proba, y_valid, k=3),
    "Hit@5": hit_at_k(valid_proba, y_valid, k=5),
    "MRR@10": mrr_at_k(valid_proba, y_valid, k=10),
    "NDCG@10": ndcg_at_k(valid_proba, y_valid, k=10),
}

# ===== 불균형 대응 지표(매크로/가중 F1) =====
metrics.update({
    "F1_macro": f1_score(y_valid, y_pred, average="macro", zero_division=0),
    "F1_weighted": f1_score(y_valid, y_pred, average="weighted", zero_division=0),
    "Precision_macro": precision_score(y_valid, y_pred, average="macro", zero_division=0),
    "Recall_macro": recall_score(y_valid, y_pred, average="macro", zero_division=0),
})

# ===== 추천 다양성(커버리지) =====
metrics.update({
    "Coverage@3 (#classes)": coverage_at_k(valid_proba, k=3),
    "Coverage@5 (#classes)": coverage_at_k(valid_proba, k=5),
})

for k, v in metrics.items():
    print(f"{k:>18}: {v}")

           LogLoss: 2.9079141912010527
        Accuracy@1: 0.38456720531128646
         TopKAcc@3: 0.5678316422397595
         TopKAcc@5: 0.6452461480646373
             Hit@3: 0.5678316422397595
             Hit@5: 0.6452461480646373
            MRR@10: 0.49218324057670165
           NDCG@10: 0.5453906548314312
          F1_macro: 0.044758620025916504
       F1_weighted: 0.32448208694421193
   Precision_macro: 0.05007338409538457
      Recall_macro: 0.046772559366719375
Coverage@3 (#classes): 315
Coverage@5 (#classes): 340


In [29]:
artifact = {
    "pipeline": final_pipe,
    "label_encoder": le,
    "feature_cols_num": feature_cols_num,
    "feature_cols_cat": feature_cols_cat,
    "seed": SEED,
}

joblib.dump(artifact, "recommend_xgb_artifact.pkl")

['recommend_xgb_artifact.pkl']